In [143]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# estilo base
sns.set_theme(style="white")

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "lines.linewidth": 1,
    "axes.linewidth": 0.8,
    "figure.dpi": 300
})

# 1 coluna
FIGSIZE_SINGLE = (7, 5)

# 2 colunas
FIGSIZE_DOUBLE = (7, 3)

#palette = sns.color_palette("colorblind")
palette = sns.color_palette("Set2") #("flare") #("Set2") #("husl", 9) #("pastel")

#palette = ["black", "dimgray", "gray"]

# pasta com os ficheiros
DATA_DIR = "./"

data = []

def parse_filename(filename):
    # remove prefixo e extensão
    name = filename.replace("results_", "").replace(".json", "")
    
    parts = name.split("_")
    
    attack = parts[0]
    type = parts[1]  # adversarial, random, etc.
    feature = None
    ratio = 1
    if len(parts) > 2:
        feature = parts[2]  # size, iat, both
        
        if len(parts) > 3:
            # extrair ratio
            ratio_part = parts[3]
            if "rest" in ratio_part:
                ratio = float(ratio_part.replace("rest", "")) / 10

    return attack, type, feature, ratio


# ler todos os ficheiros
# iterar sobre as pastas na pasta
for folder in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, folder)
    if os.path.isdir(folder_path):
        for folder2 in os.listdir(folder_path):
            folder2_path = os.path.join(folder_path, folder2)
            if os.path.isdir(folder2_path) and folder2!="Kitsune" and folder2!="doesnt work" and folder2!="old":  # ignorar pasta Kitsune por enquanto TODO
                for file in os.listdir(folder2_path):
                    if file.endswith(".json"):
                        with open(os.path.join(folder2_path, file)) as f:
                            content = json.load(f)
                    
                            attack, type, feature, ratio = parse_filename(file)

                            if type == "original":
                                data.append({
                                    "attack": attack,
                                    "feature": type,
                                    "ratio": 0,
                                    "model": content["model"],
                                    "ASR": content["metrics"]["attack_success_rate"]
                                })
                            else:
                                data.append({
                                    "attack": attack,
                                    "feature": feature,
                                    "ratio": ratio,
                                    "model": content["model"],
                                    "ASR": content["metrics"]["attack_success_rate"],
                                    "TrainLoss": content["Train_loss"]
                                })


df = pd.DataFrame(data)

# garantir ordem consistente
feature_order = ["original", "iat", "size", "both"]
df["feature"] = pd.Categorical(df["feature"], categories=feature_order, ordered=True)

df["feature"] = df["feature"].map({
    "original": "Original",
    "iat": "IAT",
    "size": "Size",
    "both": "Both"
})

ratio_order = [0, 0.2, 0.5, 1]
df["ratio"] = pd.Categorical(df["ratio"], categories=ratio_order, ordered=True)
# converter ratio para percentagem se existir
df["ratio"] = df["ratio"].apply(lambda x: f"{int(x*100)}%" if pd.notnull(x) else x)

df["TrainLoss"] = df["TrainLoss"] + 4


In [144]:
for attack in df["attack"].unique():
    # filtrar dados para o ataque atual
    subset = df[df["attack"] == attack]

    #filtrar apenas os dados com ratio 1
    subset = subset[subset["ratio"] == "100%"]

    plt.figure(figsize=FIGSIZE_SINGLE)
    
    ax = sns.barplot(
        data=subset,
        x="model",
        y="ASR",
        hue="feature",
        palette=palette,
        errorbar=None
    )

    ax.set_xlabel("Feature")
    ax.set_ylabel("ASR")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=6, padding=2)

    # remover top/right spines
    sns.despine()

    # legenda compacta
    ax.legend(title="", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.savefig(f"plots/ieee_asr_feature_{attack}.png", bbox_inches="tight")
    plt.close()

/tmp/user/1403/ipykernel_1744803/3760772131.py:10: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  ax = sns.barplot(
/tmp/user/1403/ipykernel_1744803/3760772131.py:10: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  ax = sns.barplot(
/tmp/user/1403/ipykernel_1744803/3760772131.py:10: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  ax = sns.barplot(


In [145]:
attacks = df["attack"].unique()
n = len(attacks)

fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)

# garantir que axes é iterável mesmo com 1 subplot
if n == 1:
    axes = [axes]

for ax, attack in zip(axes, attacks):
    subset = df[df["attack"] == attack]
    subset = subset[subset["ratio"] == "100%"]

    sns.barplot(
        data=subset,
        x="model",
        y="ASR",
        hue="feature",
        palette=palette,
        errorbar=None,
        ax=ax
    )

    ax.set_title(attack)
    ax.set_xlabel("Model")
    ax.set_ylabel("ASR")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=6, padding=2)

    sns.despine(ax=ax)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="", frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=len(labels))

# remover legendas individuais
for ax in axes:
    ax.get_legend().remove()

fig.subplots_adjust(wspace=1)

plt.tight_layout(w_pad=4)
plt.savefig("plots/ieee_asr_feature_all.png", bbox_inches="tight")
plt.close()

/tmp/user/1403/ipykernel_1744803/1083759058.py:14: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  sns.barplot(
/tmp/user/1403/ipykernel_1744803/1083759058.py:14: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  sns.barplot(
/tmp/user/1403/ipykernel_1744803/1083759058.py:14: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  sns.barplot(


In [146]:
for attack in df["attack"].unique():
    subset = df[df["attack"] == attack]

    subset = subset[subset["feature"] == "Size"]

    plt.figure(figsize=FIGSIZE_SINGLE)

    ax = sns.lineplot(
        data=subset,
        x="ratio",
        y="ASR",
        hue="model",
        marker="o",
        palette=palette,
        errorbar=None
    )

    ax.set_xlabel("Ratio")
    ax.set_ylabel("ASR")

    sns.despine()
    ax.legend(title="", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.savefig(f"plots/ieee_asr_ratio_{attack}.png", bbox_inches="tight")
    plt.close()

/tmp/user/1403/ipykernel_1744803/449062396.py:8: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  ax = sns.lineplot(
/tmp/user/1403/ipykernel_1744803/449062396.py:8: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  ax = sns.lineplot(
/tmp/user/1403/ipykernel_1744803/449062396.py:8: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  ax = sns.lineplot(


In [147]:
attacks = df["attack"].unique()
n = len(attacks)

fig, axes = plt.subplots(1, n, figsize=(6*n, 4), sharey=True)

if n == 1:
    axes = [axes]

for ax, attack in zip(axes, attacks):
    subset = df[df["attack"] == attack]
    subset = subset[subset["feature"] == "Size"]

    sns.lineplot(
        data=subset,
        x="ratio",
        y="ASR",
        hue="model",
        marker="o",
        palette=palette,
        errorbar=None,
        ax=ax
    )

    ax.set_title(attack)
    ax.set_xlabel("Ratio")
    ax.set_ylabel("ASR")

    sns.despine(ax=ax)

    # remover legenda individual
    ax.legend_.remove()

# legenda global (1 só para todos os subplots)
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title="Model",
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.15),
    ncol=len(labels)
)

plt.tight_layout(rect=[0, 0, 1, 0.9])

plt.savefig("plots/ieee_asr_ratio_all_attacks.png", bbox_inches="tight")
plt.close()

/tmp/user/1403/ipykernel_1744803/273243584.py:13: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(
/tmp/user/1403/ipykernel_1744803/273243584.py:13: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(
/tmp/user/1403/ipykernel_1744803/273243584.py:13: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(


In [148]:

plt.figure(figsize=FIGSIZE_SINGLE)
# print only rows where feature is IAT or Size
subset1 = df[df["ratio"]=="100%"]
subset = subset1[subset1["feature"].isin(["IAT", "Size"])]
ax = sns.scatterplot(
    data=subset,
    x="TrainLoss",
    y="ASR",
    hue="feature",
    style="feature",
    palette=palette,
)

ax.set_xlabel("Train Loss")
ax.set_ylabel("ASR")

sns.despine()
ax.legend(title="", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig("plots/ieee_asr_vs_loss.png", bbox_inches="tight")
plt.close()

/tmp/user/1403/ipykernel_1744803/2564984443.py:5: UserWarning: The palette list has more values (8) than needed (4), which may not be intended.
  ax = sns.scatterplot(


In [149]:
from matplotlib import colors
import numpy as np

xmin = subset["TrainLoss"].min()
xmax = subset["TrainLoss"].max()
ymin = subset["ASR"].min()
ymax = subset["ASR"].max()

extent = [xmin, xmax, ymin, ymax]

plt.figure(figsize=FIGSIZE_SINGLE)

subset1 = df[df["ratio"] == "100%"]
subset = subset1[subset1["feature"].isin(["IAT", "Size"])]

# separar
iat = subset[subset["feature"] == "IAT"]
size = subset[subset["feature"] == "Size"]

norm = colors.Normalize(vmin=1, vmax=5)

def truncate_colormap(cmap, minval=0.3, maxval=1.0, n=100):
    return plt.cm.colors.LinearSegmentedColormap.from_list(
        f"trunc({cmap.name})",
        cmap(np.linspace(minval, maxval, n))
    )

cmap_iat = truncate_colormap(plt.cm.YlGnBu, 0.3, 1.0)
cmap_size = truncate_colormap(plt.cm.Reds, 0.3, 1.0)

hb1 = plt.hexbin(
    iat["TrainLoss"], iat["ASR"],
    gridsize=30,
    extent=extent,
    cmap=cmap_iat,
    norm=norm,
    mincnt=1,
    alpha=0.8
)

hb2 = plt.hexbin(
    size["TrainLoss"], size["ASR"],
    gridsize=30,
    extent=extent,
    cmap=cmap_size,
    norm=norm,
    mincnt=1,
    alpha=0.8
)

cb = plt.colorbar(hb1)
cb.set_label("IAT Count", loc="bottom")

cb = plt.colorbar(hb2)
cb.set_label("Size Count", loc="bottom")


plt.tight_layout()
plt.savefig("plots/ieee_asr_vs_loss.png", bbox_inches="tight")
plt.close()

In [ ]:
from scipy.stats import pearsonr, spearmanr

for feature in ["IAT", "Size"]:
    sub = subset[subset["feature"] == feature]
    
    pearson_corr, p_p = pearsonr(sub["TrainLoss"], sub["ASR"])
    spearman_corr, p_s = spearmanr(sub["TrainLoss"], sub["ASR"])
    
    print(f"\nFeature: {feature}")
    print(f"Pearson: {pearson_corr:.3f} (p={p_p:.4f})")
    print(f"Spearman: {spearman_corr:.3f} (p={p_s:.4f})")

import statsmodels.api as sm

for feature in ["IAT", "Size"]:
    sub = subset[subset["feature"] == feature]
    
    X = sub["TrainLoss"]
    y = sub["ASR"]
    
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    print(f"\nFeature: {feature}")
    print(model.summary())

threshold = subset["TrainLoss"].median()

iat = subset[subset["feature"] == "IAT"]

low_loss = iat[iat["TrainLoss"] <= threshold]["ASR"]
high_loss = iat[iat["TrainLoss"] > threshold]["ASR"]

from scipy.stats import ttest_ind

t_stat, p_val = ttest_ind(low_loss, high_loss)

print("T-test p-value:", p_val)

# Conclusão do chat: Nada a ver


Feature: IAT
Pearson: 0.311 (p=0.2585)
Spearman: 0.314 (p=0.2541)

Feature: Size
Pearson: 0.064 (p=0.8195)
Spearman: 0.006 (p=0.9844)

Feature: IAT
                            OLS Regression Results                            
Dep. Variable:                    ASR   R-squared:                       0.097
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     1.397
Date:                Thu, 02 Apr 2026   Prob (F-statistic):              0.258
Time:                        18:42:56   Log-Likelihood:               -0.33564
No. Observations:                  15   AIC:                             4.671
Df Residuals:                      13   BIC:                             6.087
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [

/home/inespissarra/Transformer-based_Adversarial_Network_Traffic_Generator/venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/home/inespissarra/Transformer-based_Adversarial_Network_Traffic_Generator/venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [151]:
# Tables

table = df[["attack", "model", "feature", "ratio", "ASR", "TrainLoss"]]
table = table.sort_values(by=["attack", "model", "feature", "ratio"])
table.to_latex(
    "plots/results_table.tex",
    index=False,
    float_format="%.3f"
)

pivot = df.pivot_table(
    index=["attack", "feature", "ratio"],
    columns="model",
    values="ASR"
)

pivot.to_latex(
    "plots/asr_table.tex",
    float_format="%.3f"
)

pivot_loss = df.pivot_table(
    index=["attack", "feature", "ratio"],
    columns="model",
    values=["ASR", "TrainLoss"]
)

pivot_loss.to_latex(
    "plots/asr_loss_table.tex",
    float_format="%.3f"
)

/tmp/user/1403/ipykernel_1744803/2865969641.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
/tmp/user/1403/ipykernel_1744803/2865969641.py:22: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_loss = df.pivot_table(
